# 📘 Deploy llama.cpp Inference Service
> **Applicable Environment**: Kubernetes Pod (Ubuntu base image, no Docker/Podman, AMD ROCm environment)
> **Purpose**: Compile llama.cpp (HIP/ROCm acceleration), download GGUF models, start and monitor OpenAI-compatible inference service.
> **Prerequisites**: First run `1_init_pod_env_cn.md` to complete `/data` symlink; GPU drivers and ROCm tools (`rocminfo` / `amd-smi` / `hipconfig`) are ready.

## 1. Service Overview

- **Source/Binaries**: `/data/app/llama.cpp` (`bin/llama-server` etc. 120 files, including `libggml-hip.so`)
- **Config Scripts**: `/data/service/llamacpp/scripts/` (builder / download / server)
- **Model Directory**: `/data/data-store/llamacpp/models/`
- **Log Directory**: `/data/data-store/llamacpp/logs/`
- **Tuning Documentation**: `/data/service/llamacpp/docs/` (7B / 30B / 80B config and benchmarks)

### Local Hardware (Reference docs and actual measurements)
| Component | Specs |
|------|------|
| GPU | AMD Radeon gfx1100 (RDNA3), 96 CU, 48 GB VRAM |
| CPU | AMD EPYC 9334, 2 Socket, 128 vCPU |
| RAM | ~503 GB |
| ROCm | 7.2.1 |
| llama.cpp | HIP build (`-DGGML_HIP=ON`) |

### Default Model: Qwen3-Coder-30B-A3B (Q4_K_M)
- Repository: `lucataco/Qwen3-Coder-30B-A3B-Instruct-Q4_K_M-GGUF` (~18.6 GB)
- Config: 512K context / 6 concurrent slots / KV `q4_0` / full GPU layers (measured ~90 tok/s, VRAM 32.3/48 GB)

---

## 2. Clone Source Code

Clone llama.cpp source code to persistent directory `/data/app` (survives Pod restarts, no need to re-clone).

```python
%%bash
#!/bin/bash
set -euo pipefail

# Determine sudo
SUDO=""
if [ "$(id -u)" -ne 0 ]; then
    if command -v sudo &>/dev/null && sudo -n true 2>/dev/null; then
        SUDO="sudo -n"
    else
        echo "❌ Not running as root and sudo not available. Please run as root or grant passwordless sudo."
        exit 1
    fi
fi

SRC="/data/app/llama.cpp"

# Ensure git is installed
if ! command -v git &>/dev/null; then
    echo "🔧 Installing git..."
    $SUDO apt-get update -qq
    $SUDO apt-get install -y -qq git
fi

if [ -d "$SRC/.git" ]; then
    echo "✅ llama.cpp source already exists: $SRC"
    git -C "$SRC" log --oneline -1
else
    echo "🔧 Cloning llama.cpp ..."
    mkdir -p /data/app
    git clone https://github.com/ggml-org/llama.cpp.git "$SRC"
    echo "✅ Clone complete: $SRC"
fi
```

---

## 3. Build (llamacppbuilder.sh)

Use `/data/service/llamacpp/scripts/llamacppbuilder.sh` to build; auto-detects GPU (`gfx1100`) and enables HIP acceleration.

**Script Highlights**
- Default CMake: `-DGGML_HIP=ON -DGGML_HIP_ROCWMMA_FATTN=ON -DGGML_HIP_NO_VMM=ON -DGPU_TARGETS=<auto>`
- `--hackathon` mode adds: LTO, HIP graphs, rocWMMA flash attention, native CPU (AVX2/FMA/BMI2), ccache
- Output directory: `<source>/build` (default `/data/app/llama.cpp/build`)

```python
%%bash
#!/bin/bash
set -euo pipefail

BUILDER="/data/service/llamacpp/scripts/llamacppbuilder.sh"
SRC="/data/app/llama.cpp"

# Check if binary already exists
if [ -x "$SRC/bin/llama-server" ]; then
    echo "✅ llama-server already built: $($SRC/bin/llama-server --version | head -1)"
    exit 0
fi

# Ensure builder script exists
if [ ! -x "$BUILDER" ]; then
    echo "❌ Builder script not found: $BUILDER"
    exit 1
fi

# Detect GPU target
detect_gpu() {
    rocminfo 2>/dev/null | grep -m1 'Name:.*gfx' | awk '{print $2}' || echo "gfx1100"
}
echo "GPU target: $(detect_gpu)"

# Build (hackathon mode)
echo "🔧 Building llama.cpp with HIP (this may take 10~30 minutes)..."
bash "$BUILDER" --hackathon

# Move build/bin to source root for convenience
if [ -d "$SRC/build/bin" ]; then
    echo "🔧 Moving build/bin -> $SRC/bin ..."
    mv "$SRC/build/bin" "$SRC/"
fi

# Verify
if [ -x "$SRC/bin/llama-server" ]; then
    echo "✅ Build successful: $SRC/bin/llama-server"
    ls -la "$SRC/bin" | grep -E "llama-server|llama-cli|llama-embedding"
else
    echo "❌ Build failed or binary not found"
    exit 1
fi
```

> 💡 If binaries already exist, the build step is skipped automatically.

---

## 4. Install Model Download Tool

`download_model.sh` depends on HuggingFace CLI `hf` (huggingface_hub). For domestic networks, it is recommended to use mirror `https://hf-mirror.com` (enabled by default in script).

```python
%%bash
#!/bin/bash
set -euo pipefail

# Determine sudo for pip
SUDO=""
if [ "$(id -u)" -ne 0 ]; then
    if command -v sudo &>/dev/null && sudo -n true 2>/dev/null; then
        SUDO="sudo -n"
    else
        echo "❌ Not running as root and sudo not available."
        exit 1
    fi
fi

if ! command -v hf &> /dev/null; then
    echo "🔧 Installing huggingface_hub ..."
    # Try --break-system-packages if supported, otherwise fallback to --user
    if pip install --help | grep -q break-system-packages; then
        $SUDO pip install --break-system-packages -q huggingface_hub
    else
        $SUDO pip install -q huggingface_hub || pip install --user -q huggingface_hub
    fi
    echo "✅ Install complete"
fi
hf --version
```

---

## 5. Download Model (download_model.sh)

```python
%%bash
#!/bin/bash
set -euo pipefail

MODEL_DIR="/data/data-store/llamacpp/models/Qwen3-Coder-30B-A3B-Q4_K_M"
MODEL_FILE="$MODEL_DIR/qwen3-coder-30b-a3b-instruct-q4_k_m.gguf"

# Check if model already exists
if [ -f "$MODEL_FILE" ]; then
    echo "✅ Model already exists: $MODEL_FILE ($(du -h $MODEL_FILE | cut -f1))"
    exit 0
fi

SCRIPT="/data/service/llamacpp/scripts/download_model_30b.sh"
if [ ! -x "$SCRIPT" ]; then
    echo "❌ Download script not found: $SCRIPT"
    exit 1
fi

# Download 30B Q4_K_M (using mirror by default)
echo "🔧 Downloading 30B Q4_K_M model (~18.6GB) ..."
bash "$SCRIPT" 30b

# Verify after download
if [ -f "$MODEL_FILE" ]; then
    echo "✅ Download successful: $MODEL_FILE ($(du -h $MODEL_FILE | cut -f1))"
else
    echo "❌ Download failed or model not found"
    exit 1
fi
```

> Note: For 30B Q4_K_M use `download_model_30b.sh`; `download_model.sh`'s `30b` is Q8_0 version.

After download, models are located at:

| Model | Path |
|------|------|
| 30B Q4_K_M | `/data/data-store/llamacpp/models/Qwen3-Coder-30B-A3B-Q4_K_M/qwen3-coder-30b-a3b-instruct-q4_k_m.gguf` |
| 80B Q4_K_M | `/data/data-store/llamacpp/models/Qwen3-Coder-Next-Opus-Distilled-Q4_K_M/*.gguf` |

> ✅ Unified: `llamaserver.sh` default `BASE_MODEL_DIR=/data/data-store/llamacpp/models`, consistent with download script directory, no extra settings needed.

---

## 6. Start Service (llamaserver.sh)

Unified management script: `start / stop / status / test`, supports `--model 30b`, `--port`, `--verbose`, `--dry-run`.

**Startup parameters (30B)**: `-ngl -1` (full GPU), `-c 524288` (512K), `-np 6`, KV `q4_0`, `--flash-attn on`, `--jinja`, `--numa distribute`.

```python
%%bash
#!/bin/bash
set -euo pipefail

SCRIPT="/data/service/llamacpp/scripts/llamaserver.sh"
if [ ! -x "$SCRIPT" ]; then
    echo "❌ llamaserver.sh not found: $SCRIPT"
    exit 1
fi

# Check if already running
if pgrep -f "llama-server" >/dev/null; then
    echo "✅ llama-server is already running"
    bash "$SCRIPT" status
    exit 0
fi

# Start 30B (default model directory already consistent with download script)
echo "🚀 Starting llama-server with 30B model..."
bash "$SCRIPT" start --model 30b

# Wait a few seconds and check status
sleep 5
bash "$SCRIPT" status
```

**Verify readiness**:

```python
%%bash
#!/bin/bash
set -euo pipefail

# Health check (returns "ok" when ready)
if curl -s --max-time 5 http://localhost:8080/health | grep -q ok; then
    echo "✅ Service is healthy"
else
    echo "⚠️  Health check failed (service may still be loading the model)"
    echo "   Check logs: tail -f /data/data-store/llamacpp/logs/llamaserver-*.log"
fi
```

---

## 7. Inference Testing

```python
%%bash
#!/bin/bash
set -euo pipefail

# One-click test (health check + OpenAI-compatible inference + speed/token stats)
bash /data/service/llamacpp/scripts/llamaserver.sh test

# Or directly call OpenAI-compatible endpoint
curl -s --max-time 60 http://localhost:8080/v1/chat/completions \
    -H "Content-Type: application/json" \
    -d '{"model":"test","messages":[{"role":"user","content":"Please introduce yourself in one sentence"}],"max_tokens":50,"temperature":0.7}' \
    | python3 -m json.tool --no-ensure-ascii
```

**Common Endpoints**

| Endpoint | Description |
|------|------|
| `GET /health` | Health check |
| `GET /v1/models` | Model list |
| `POST /v1/chat/completions` | Chat completion |
| `POST /v1/embeddings` | Embedding (requires `--embedding`) |
| `GET /props` | Runtime parameter details |

---

## 8. Monitoring (monitor.sh)

`/data/service/monitor.sh`: Text-based monitoring panel refreshed every 10 seconds, showing llama.cpp (8080), unires backend (8000), frontend (7860) status, and CPU / memory / GPU utilization and VRAM.

```python
%%bash
#!/bin/bash
set -euo pipefail

# Run monitoring panel (exit with Ctrl+C)
# bash /data/service/monitor.sh

# Manual quick view
echo "== llama-server process =="
pgrep -a llama-server || echo "Not running"
echo
echo "== GPU VRAM =="
if command -v amd-smi &>/dev/null; then
    amd-smi metric --mem 2>/dev/null | grep -E "TOTAL_VRAM|USED_VRAM" | head -4
elif command -v rocm-smi &>/dev/null; then
    rocm-smi --showmeminfo vram 2>/dev/null
else
    echo "No AMD monitoring tool found"
fi
```

---

## 9. Service Management and Logs

```python
%%bash
#!/bin/bash
set -euo pipefail

# Stop (graceful SIGTERM, force kill on timeout; double fallback by PID file + port)
# bash /data/service/llamacpp/scripts/llamaserver.sh stop

# Logs
LOG_DIR=/data/data-store/llamacpp/logs
ls -la "$LOG_DIR" 2>/dev/null || echo "（No logs yet, service has never been started）"
```

**Common Operations Commands**
```bash
tail -f /data/data-store/llamacpp/logs/llamaserver-qwen3.log   # Tail logs
ss -tlnp | grep 8080                                          # Port occupation
rocm-smi --showmeminfo vram                                   # VRAM usage
```

**One-click Pod restart recovery** (similar to PostgreSQL's `init-pg.sh`):
```bash
/data/init/init-llamacpp.sh   # Install dependencies → compile if binaries missing → download if model missing → start (wait up to 10 minutes)
```

---

## 10. Troubleshooting

| Issue | Solution |
|------|------|
| `hf` command not found | `pip install --break-system-packages huggingface_hub` |
| Download network unreachable | Script defaults to `HF_ENDPOINT=https://hf-mirror.com`, or use `--no-mirror` |
| Model not found | Check `BASE_MODEL_DIR` matches download directory (see Section 5) |
| Startup timeout/failure | `cat /data/data-store/llamacpp/logs/llamaserver-*.log` |
| First startup very slow | Model is on PVC network storage, 18.6GB read/upload to VRAM takes 5~10 minutes, normal; `status` shows process alive, just wait |
| VRAM insufficient | Reduce context `-c`, concurrency `-np`, or KV quantization (`q4_0`) |
| Segmentation fault/illegal instruction | Confirm binary is HIP build and `GPU_TARGETS` is correct (`gfx1100`) |
| Build slow | Use `--hackathon` or reduce `-j`; first run `cmake --build build -j 128` |

For tuning details, see `/data/service/llamacpp/docs/01_7B_model_config.md` ~ `05_tuning_history.md` and `scripts/llama-server-params.md`.

---

## 11. Next Steps

- Embedding: For RAG scenarios, use `--embedding` (or refer to document #3 for pgvector vector storage).
- Performance benchmark: Use `llamaserver.sh test` to record tok/s; compare against `docs/04_30B_tuning_benchmark.md`.
- Persistence: Models are already on PVC (`/data/data-store`), no re-download needed after Pod restart, just run `llamaserver.sh start`.
- Monitoring: Use `/data/service/monitor.sh` to observe PostgreSQL, unires backend/frontend uniformly.

---

> ✅ At this point, llama.cpp inference service deployment is complete; can be integrated into Uni-Resource Agent application (refer to `2_app_cn.md`).